# Разработка модели машинного обучения

## Импортирование библиотек

In [22]:
# для работы с датафреймами
import pandas as pd

# для работы с массивами
import numpy as np

# для преобразования текста
from nltk.tokenize import sent_tokenize

# вспомогательные функции
from function import *

# для работы с датасетами
from datasets import Dataset, DatasetDict

# токенизатор и модель
from transformers import T5Tokenizer, T5ForConditionalGeneration
# аргументы для обучения, и трейнер
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
# коллатор
from transformers import DataCollatorForSeq2Seq

# основной модуль для нейронных сетей
import torch

# модуль с метрикой оценивания
import evaluate
# модуль для модели сравнения
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
# модуль для разбиения предложений на слвоа
from nltk.tokenize import word_tokenize
# косинусное сходство
from sklearn.metrics.pairwise import cosine_similarity
# для работы с векторами
import numpy as np


Определяю устройство, на котором будут производиться вычисления:

In [3]:
# получаю девайс
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# вывожу девайс
print(device)

cuda


Описание вспомогательных функций *(реализованных в файле `function.py`)*

```py
# функция для сравнения двух текстов
def get_similarity(text1: str, 
                   text2: str, 
                   model: str | Doc2Vec, 
                   prep_flag: bool = True, 
                   preprocess: Callable[[str], str] = get_input):
    '''
    Функция для получения схожести двух текстов
    ===
        Args:
            - text1 (str): первый текст в строковом формате
            - text2 (str): второй текст в строковом формате
            - model (str|Doc2Vec): модель в формате doc2vec объекта 
                                  либо путь к ней в строковом формате

        Returns:
            - float: сходство между текстами (от 1 до -1)
    '''
    # проверка формата текста 1
    if not isinstance(text1, str):
        raise TypeError(f'text1 должен быть в строковом формате, а не {type(text1)}')
    # проверка формата текста 2
    if not isinstance(text1, str):
        raise TypeError(f'text1 должен быть в строковом формате, а не {type(text1)}')
    # проверка формата модели
    if not isinstance(model, (str, Doc2Vec)):
        raise TypeError(f'model должна быть в формате Doc2Vec, либо в виде строкового путя, а не {type(model)}')
    # если модель в виде путя
    if type(model) == str:
        # проверяем, что файл существует
        if os.path.exists(model):
            # пробуем загрузить модель через try: except
            try:
                # загружаем модель из файла
                model = Doc2Vec.load(model)
            # если не получилось загрузить, выводим ошибку
            except Exception as e:
                raise ValueError(f'Ошибка! не удалось загрузить модель, проверьте ваш файл!\n{e}')
        # если путь не существует:
        else:
            # выводим ошибку
            raise ValueError('Ошибка! Файла с моделью не существует, проверьте правильность написания!')
    # если стоит метка о обработке данных
    if prep_flag:
        text1 = preprocess(text1)
        text2 = preprocess(text2)
    # вычисляем эмбеддинг
    inferred_vector1 = model.infer_vector(word_tokenize(text1.lower())).reshape(1,-1)
    inferred_vector2 = model.infer_vector(word_tokenize(text2.lower())).reshape(1,-1)
    # # получаем сходство
    return cosine_similarity(inferred_vector1, inferred_vector2).item()
```

```py
# функция для поиска топ 3 схожих статей
def find_top_similar(df: pd.DataFrame, text: str, model: Doc2Vec, top_n: int = 3) -> pd.DataFrame:
    '''
    Находит топ-N наиболее схожих эмбеддингов и их summary.

    Параметры:
        df (pd.DataFrame): Датафрейм с колонками 'summary' и 'embedding'.
        input_embedding (np.ndarray): Входной эмбеддинг для сравнения.
        top_n (int): Количество наиболее схожих результатов (по умолчанию 3).

    Возвращает:
        tuple
    '''
    # получаем эмбеддинг
    input_embedding = model.infer_vector(word_tokenize(get_input(text))).reshape(1,-1)
    # список всех схожестей
    similarities = []
    # проходимся по всему датафрейму
    for i in range(df.shape[0]):
        similarities.append(cosine_similarity(input_embedding, df['embedding'].loc[i]).item()*100)
    # Добавляем столбец с косинусной схожестью в датафрейм
    df['similarity'] = similarities
    
    # Сортируем датафрейм по убыванию схожести и выбираем топ-N
    top_similar = df.sort_values(by='similarity', ascending=False).head(3)
    summaries = top_similar['summary'].tolist()
    top_similarities = [round(sim, 2) for sim in top_similar['similarity'].tolist()]
    headers = [' '.join(word_tokenize(summary)) for summary in summaries]

    # Возвращаем только нужные колонки
    return top_similarities, headers
```

```py
# функция для извлечения эмбеддингов из всего датасета
def extract_all_embeddings(dataset, part, column, model: Doc2Vec):
    df = pd.DataFrame(columns=['summary', 'embedding'])
    # проходимся по каждой суммаризации в датасете
    for i in tqdm(range(len(dataset[part])), desc='Извлечение эмбеддингов..', unit='text'):
        # получаем суммаризацию
        summary = dataset[part][column][i]
        # получаем эмбеддинг
        embedding = model.infer_vector(word_tokenize(get_input(summary))).reshape(1,-1)
        # сохраняем эмбеддинг
        df.loc[i] = [summary, embedding]
    df.head()
```

## Загрузка данных

In [4]:
# загружаю тренировочную выборку
train_dataset = Dataset.from_parquet("Dataset/train.parquet")
# валидационную выборку
validation_dataset = Dataset.from_parquet("Dataset/validation.parquet")
# тестовую выборку
test_dataset = Dataset.from_parquet("Dataset/test.parquet")
# загружаю тренировочную выборку
train_dataset = Dataset.from_parquet("Dataset/train.parquet")
# валидационную выборку
validation_dataset = Dataset.from_parquet("Dataset/validation.parquet")
# тестовую выборку
test_dataset = Dataset.from_parquet("Dataset/test.parquet")

# объединяю все выборки в объект Dataset
dataset = DatasetDict({
    "train": train_dataset,
    "validation": validation_dataset,
    "test": test_dataset
})
# вывожу структуру получившегося набора данных
dataset

DatasetDict({
    train: Dataset({
        features: ['text_path', 'annotation_path', 'tags_path', 'text', 'summary', 'tag', 'text_all_symb', 'summary_all_symb', 'tag_all_symb', 'text_clean', 'summary_clean', 'tag_clean', 'text_words', 'summary_words', 'tag_words', 'id', 'processed_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 329
    })
    validation: Dataset({
        features: ['text_path', 'annotation_path', 'tags_path', 'text', 'summary', 'tag', 'text_all_symb', 'summary_all_symb', 'tag_all_symb', 'text_clean', 'summary_clean', 'tag_clean', 'text_words', 'summary_words', 'tag_words', 'id', 'processed_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 41
    })
    test: Dataset({
        features: ['text_path', 'annotation_path', 'tags_path', 'text', 'summary', 'tag', 'text_all_symb', 'summary_all_symb', 'tag_all_symb', 'text_clean', 'summary_clean', 'tag_clean', 'text_words', 'summary_words', 'tag_words', 'id', 'processed_text', 'input_ids', 

## Подбор алгоритма обучения

### Модель суммаризации

Передо мной стоит задача `суммаризации` текста. Это значит, что надо реализовать нейронную сеть, которая будет находить и выписывать краткое содержание текста. Для этого я буду использовать предобученную модель `"sarahai/ruT5-base-summarizer"`, которую настрою работать на своих данных

Модель `RuT5` адаптирована для работы с русским языком, что важно для корректной обработки и генерации текста на русском

Архитектура `T5` (`Text-To-Text Transfer Transformer`) - это архитектура которая хорошо справляется с задачами преобразования текста, включая суммаризацию

Использование предобученной модели позволяет сэкономить время и ресурсы, так как модель уже обучена на большом объеме данных, и может быть дообучена (`fine-tunned`) на собственном наборе данных для решения более конкретной задачи

### Модель сравнения

Для `сравнения` текстов, я буду использовать `эмбеддинги` текста - векторные представления, а для извлечения `эмбеддингов` я буду использовать модель `Doc2Vec`

`Doc2Vec` преобразует тексты в `вектора`, что позволяет сравнивать их между собой (в моем случае я буду использовать метрику `косинусного сходства`)

В отличие от других, более простых методов, `Doc2Vec` учитывает `семантику` и `контекст` слов, что делает сравнение более точным

Благодаря тому, что я буду обучать модель на своем `корпусе данных`, модель сможет более точно сравнивать именно тексты заданного формата (`статьи`)

### Метрики для суммаризации текста

Для суммаризации одной из наиболее часто используемых метрик является оценка `ROUGE` (сокращение от `Recall-Oriented Understudy for Gisting Evaluation`). Основная идея этой метрики заключается в сравнении сгенерированного текста с набором эталонных текстов, которые обычно создаются людьми. Чтобы сделать ее более точной, предположим, что мы хотим сравнить следующие две строчки:

In [5]:
generated_summary = "I absolutely loved reading the Hunger Games"
reference_summary = "I loved reading the Hunger Games"

Одним из способов их сравнения может быть подсчет `количества перекрывающихся слов`, которых в данном случае будет 6. Однако это несколько грубовато, поэтому вместо этого `ROUGE` основывается на вычислении оценок `precision` и `recall` для перекрытия

Для `ROUGE` `recall` измеряет, насколько эталонное резюме соответствует `сгенерированному`. Если мы просто сравниваем слова, `recall` можно рассчитать по следующей формуле:

$$\text{Recall} = \frac{\text{Number of overlapping words}}{\text{Total number of words in reference summary}}$$

Для нашего простого примера выше эта формула дает идеальный `recall` 6/6 = `1`; то есть все слова в эталонном тексте были получены моделью. Это может показаться замечательным, но представьте, если бы сгенерированный нами текст был “I really really loved reading the Hunger Games all night”. Это тоже дало бы идеальный `recall`, но, возможно, было бы хуже, поскольку было бы многословным. Чтобы справиться с этими сценариями, мы также вычисляем `precision`, которая в контексте `ROUGE` измеряет, насколько `сгенерированное` резюме было `релевантным`:

$$\text{Precision} = \frac{\text{Number of overlapping words}}{\text{Total number of words in generated summary}}$$

Если применить это к нашему подробному тексту, то `precision` составит 6/10 = 0,6, что значительно хуже, чем `precision` 6/7 = 0,86, полученная при использовании более короткого текста. На практике обычно вычисляют и `precision`, и `recall`, а затем `F1-score` (среднее гармоническое из `precision` и `recall`)

Загружаю метрику `ROUGE`:

In [6]:
# загружаю используемую метрику
rouge_score = evaluate.load('rouge')

Затем мы можем использовать функцию `rouge_score.compute()`, чтобы рассчитать все метрики сразу:

In [9]:
# считаем метрики примеров
scores = rouge_score.compute(
    predictions=[generated_summary],
    references=[reference_summary]
)
# вывожу получившиеся метрики
scores

{'rouge1': 0.923076923076923,
 'rouge2': 0.7272727272727272,
 'rougeL': 0.923076923076923,
 'rougeLsum': 0.923076923076923}

## Работа с нейронной сетью

#### Модель суммаризации

Для начала, надо инициализировать саму `модель` и ее `токенизатор`:

In [7]:
# имя модели
model_name = "sarahai/ruT5-base-summarizer"
# инициализируем модель
model = T5ForConditionalGeneration.from_pretrained(model_name)
# инициализирую токенизатор
tokenizer = T5Tokenizer.from_pretrained(model_name)

Теперь, надо назначить вычислительное устройство для модели (его я определил выше):

In [8]:
# назначаю устройство и вывожу архитектуру модели
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

Дообучение `ruT5` с `API` `Trainer`

In [9]:
# назначаю кол-во батчей
batch_size = 8
# кол-во эпох
epochs = 30
# функция потерь
logging_steps = len(dataset['train']) // batch_size
name = model_name.split('/')[-1]
print(name)

ruT5-base-summarizer


Теперь, нужно создать объект `Seq2SeqTrainingArguments`, чтобы заполнить гиперпараметры модели:

In [ ]:
# объект с гиперпараметрами
args = Seq2SeqTrainingArguments(
    output_dir='model-4-summary',               # название папки для сохранения модели
    overwrite_output_dir=True,                  # переписывать существующие сохраненные модели
    eval_strategy='epoch',                      # оценка после каждой эпохи
    per_device_train_batch_size=batch_size,     # кол-во тренировочных батчей
    per_device_eval_batch_size=batch_size,      # кол-во батчей для оценки
    gradient_accumulation_steps=2,              # кол-во шагов накопления градиента до обновления
    torch_empty_cache_steps=4,                  # oчистка кэша GPU через каждые 4 шага
    learning_rate=1e-4,                         # скорость обучения
    num_train_epochs=epochs,                    # кол-во эпох
    logging_steps=logging_steps,                # частота логов
    seed=42,                                    # сид для воспроизводимости результатов
    fp16=True,                                  # использование mixed precision для ускорения обучения
    weight_decay=0.01,                          # отложенные весаL2-регуляризация для предотвращения переобучения
    optim='adamw_torch',                        # оптимизатор
)

Следующее, что нужно сделать, это предоставить тренеру функцию `compute_metrics()`, чтобы оценить нашу модель во время обучения. Для суммаризации это немного сложнее, чем просто вызвать `rouge_score.compute()` для прогнозов модели, поскольку нужно декодировать выводы и метки в текст, прежде чем вычислить оценку `ROUGE`. Следующая функция делает именно это, а также использует функцию `sent_tokenize()` из `nltk` для разделения предложений резюме символом новой строки

In [ ]:
# функция для вычисления метрик
def compute_metrics(eval_pred):
    # получаем предсказания и их метки
    predictions, labels = eval_pred
    # Если predictions — это кортеж, берем первый элемент (логи)
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Преобразуем логи в индексы токенов с помощью argmax
    predicted_token_ids = np.argmax(predictions, axis=-1)
    # декодируем сгенерированные суммаризации в текст
    decoded_preds = tokenizer.batch_decode(predicted_token_ids, skip_special_tokens=True)
    # заменяем -100 в метках, тк декодировать их нельзя
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    # декодируем эталонные изложения 
    decoded_summary = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # ROUGE ожидает символ новой строки после каждого предложения
    decoded_preds = ['\n'.join(sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_summary = ['\n'.join(sent_tokenize(label.strip())) for label in decoded_summary]
    
    # вычисляем метрики ROUGE
    result = rouge_score.compute(
        predictions=decoded_preds,
        references=decoded_summary,
        use_stemmer=True            # проверить и с ним, и без него
    )

    # получаем оценки
    result = {k: v*100 for k, v in result.items()}
    return {k: round(v,4) for k,v in result.items()}

Также, для обучения модели необходим `коллатор`, который будет сдвигать метки на 1 каждый шаг

In [12]:
# инициализируем коллатор
collator = DataCollatorForSeq2Seq(tokenizer, model)

Получаем датасет для обучения модели

In [13]:
# убираем лишние колонки
train_data = dataset.select_columns(['input_ids', 'attention_mask', 'labels'])
# выводим датасет
train_data

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 329
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 41
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 42
    })
})

Теперь, надо создать объект `Trainer` для запуска обучения модели

In [15]:
# объект trainer 
trainer = Seq2SeqTrainer(
    model,                                  # модель
    args,                                   # тренировочные аргументы
    train_dataset=train_data['train'],      # датасет для обучения модели
    eval_dataset=train_data['validation'],  # датасет для оценки модели
    data_collator=collator,                 # коллатор
    processing_class=tokenizer,             # токенизатор
    compute_metrics=compute_metrics         # функция для вычисления метрик
)

Очищаю кэш видеокарты, чтобы разгрузить память

In [5]:
torch.cuda.empty_cache()

После этого, можно начать обучение модели

In [17]:
# запуск обучения модели
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,No log,1.661759,11.037000,2.570900,11.012300,11.070800
2,1.826200,1.642799,11.373600,2.439000,11.406800,11.441800
3,1.826200,1.636124,9.957300,2.623800,9.823600,9.897200
4,1.424400,1.635471,10.178300,2.623800,10.250300,10.183900
5,1.424400,1.657753,11.335700,2.702700,11.405900,11.400800
6,1.194900,1.663106,11.655000,2.695800,11.722200,11.665400
7,1.194900,1.701361,13.588100,2.664200,13.450500,13.616400
8,1.029700,1.762112,12.066900,2.439000,11.648200,12.085700
9,1.029700,1.774765,14.967100,3.353700,14.342600,14.676300
10,0.879100,1.787301,13.137000,3.515700,12.565900,12.880200


TrainOutput(global_step=630, training_loss=0.7540824451143779, metrics={'train_runtime': 5274.217, 'train_samples_per_second': 1.871, 'train_steps_per_second': 0.119, 'total_flos': 6010414379827200.0, 'train_loss': 0.7540824451143779, 'epoch': 30.0})

Сохраняю полученную модель

In [18]:
# Сохранение модели
model.save_pretrained("./saved_model")

# Сохранение токенизатора
tokenizer.save_pretrained("./saved_model")

('./saved_model\\tokenizer_config.json',
 './saved_model\\special_tokens_map.json',
 './saved_model\\spiece.model',
 './saved_model\\added_tokens.json')

#### Модель для сравнения

Собираю все тренировочный суммаризации в один список

In [8]:
summary_data = dataset['train']['summary']
print(len(summary_data))

329


In [9]:
text_data = dataset['train']['processed_text']
print(len(text_data))

329


Очищаю кэш

In [25]:
torch.cuda.empty_cache()

Теперь, я пройдусь по всем известным текстам, и выведу для них суммаризацию 

In [10]:
summaries = []

# Создаем прогресс-бар с начальным описанием
memory = torch.cuda.memory_allocated()
progress_bar = tqdm(text_data, desc=f"Memory allocated: {memory / 1e9:.2f} GB", unit="text")
for text in progress_bar:
    # Память до генерации
    memory = torch.cuda.memory_allocated()
    
    summary = get_summary(text, tokenizer, model, device)
    summaries.append(summary)
    
    progress_bar.set_description(
        f"Memory allocated: {memory / 1e9:.2f} GB"
    )
    
    torch.cuda.empty_cache()

Memory allocated: 0.99 GB: 100%|██████████| 329/329 [22:52<00:00,  4.17s/text]  


In [11]:
summary_data = summary_data + summaries

In [12]:
# создаем список для хранения данных
data = []
# проходимся по каждому тексту в датасете
for i in tqdm(range(len(summary_data)), desc='Собираем данные..', unit='text'):
    # извлекаем текст
    data.append(get_input(summary_data[i]))
    

Собираем данные..: 100%|██████████| 658/658 [09:14<00:00,  1.19text/s]


Это надо было сделать для того, чтобы искусственно увеличить тренировочную выборку, и лучше обучить модель

Сохраняю список суммаризации в текстовый файл, на всякий случай

In [ ]:
# Открываем файл для записи
with open("data_summary.txt", "w", encoding="utf-8") as file:
    file.write("\n\n".join(summary_data))  # Объединяем элементы списка в строку

Теперь, токенизирую собранные данные

In [13]:
# токенизируем данные
tokenized_data = [word_tokenize(summary.lower()) for summary in data]

Создаю объект `TaggedDocument`, который нужен для передачи данных в модель

In [14]:
# создаем объект TaggedDocument
tagged_data = [TaggedDocument(words=words, tags=[str(idx)])
               for idx, words in enumerate(tokenized_data)]


Обучение модели

In [ ]:
# тренируем модель Doc2Vec
model_sim = Doc2Vec(vector_size=100,        # размер вектора слова или документа
                window=2,                   # размер контекстного окна
                min_count=2,                # минимальная частота слова (игнорирует все, которые встречаются реже указанного значения)
                workers=4,                  # количество потоков
                epochs=100,                 # количество эпох
                dm=0,                       # алгоритм обучения (PV-DM)
                dbow_words=1,               # обучать векторы слов
                alpha=0.025,                # начальная скорость обучения
                min_alpha=0.001,            # минимальная скорость обучения
                negative=20,                # колво шумовых слов
                sample=1e-5,                # порог отсечения частых слов
                seed=42,                    # фиксация сида
                hs=1                        # иерархический софтмакс
                )
model_sim.build_vocab(tagged_data)
model_sim.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)
# сохранение
model_sim.save('doc2vec.model')

Нейронная сеть для сравнения успешно обучена

## Анализ эффективности модели

### Суммаризация

Загружаю модель из файла

In [6]:
# загрузка модели
model = T5ForConditionalGeneration.from_pretrained("./saved_model")

# загрузка токенизатора
tokenizer = T5Tokenizer.from_pretrained("./saved_model")

Далее, опять назначаю устройство вычисление для модели

In [7]:
# привожу модель к устройству
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [ ]:
# индекс строки с примером из датасета
index = 4
# оригинальные текст
orig_text = dataset['validation']['text'][index]
# оригинальная суммаризация
summary = dataset['validation']['summary'][index]
# выводим суммаризацию
print(summary)

Используя реализованную в файле `function.py` функцию `get_summary`, попробую получить суммаризацию данного текста:

In [ ]:
# вывожу суммаризацию
generated_sum = get_summary(text=orig_text, 
            tokenizer=tokenizer, 
            model=model, 
            device=device, 
            show_output=True)    # включаю автоматический вывод результатов

### Сравнение

Загружаю модель из файла:

In [ ]:
# загрузка
model_sim = Doc2Vec.load('doc2vec.model')

Два пробных текста *(изначальная суммаризация и сгененированная)*

In [ ]:
# получаем 2 суммаризации
text1 = summary_data[1]
text2 = summary_data[330]

Для того чтобы сравнить текста, надо получить эмбеддинги

In [ ]:
# вычисляем эмбеддинг
inferred_vector1 = model_sim.infer_vector(word_tokenize(get_input(text1))).reshape(1,-1)
inferred_vector2 = model_sim.infer_vector(word_tokenize(get_input(text2))).reshape(1,-1)

Для вычисления схожести я буду использовать функцию `get_similarity` реализованную в файле `function.py`

In [ ]:
# получаем сходство
similarity = get_similarity(text1, text2, model_sim)
similarity

0.8784244060516357

Используя функцию `extract_all_embeddings` выгружаю эмбеддинги всех суммаризаций

In [ ]:
extract_all_embeddings(dataset, 'train', 'summary', model_sim)

Извлечение эмбеддингов..: 100%|██████████| 329/329 [05:01<00:00,  1.09text/s]


In [ ]:
embs = pd.read_pickle('all_embs.pkl')
embs.head()

,summary,embedding
0,"В данной статье указаны основные вопросы, кото...","[[0.7675859, 0.3641603, 0.95482713, -0.2786018..."
1,Серверные приложения являются критически важны...,"[[0.60429025, -0.579292, 0.6794108, 0.42171627..."
2,Обеспечение высокого уровня доступности интерн...,"[[0.23004857, 0.08845289, -0.056778233, 0.1922..."
3,В работе были проанализированы подходы к оптим...,"[[0.5406296, 0.053741835, 0.11830298, 0.226512..."
4,ѕроводитс€ анализ возможностей использовани€\n...,"[[0.17076421, -0.039292604, 0.34421498, 0.4596..."


для вывода топ 3 самых похожих текстов из датафрейма я буду использовать функцию `find_top_similar` из файла `function.py`

In [ ]:
find_top_similar(embs, text1, model_sim)

array([[-0.02756682,  0.06266876, -0.00674738,  0.11523352, -0.2769106 ,
         0.36253974,  0.02930231,  0.04999359, -0.00684423,  0.2039667 ,
        -0.05091163,  0.15339978,  0.16296284,  0.29639333,  0.07788314,
        -0.09428301, -0.31424195, -0.13316733, -0.14294015, -0.02573602,
        -0.41086963, -0.2139987 ,  0.34184104,  0.14831117,  0.05626034,
         0.3023472 , -0.233654  ,  0.2649084 ,  0.06854548, -0.08708176,
        -0.06655807, -0.06988961, -0.04154947,  0.06200563,  0.12485714,
        -0.17591733,  0.20824231, -0.15796858, -0.37414834, -0.20735608,
         0.25030366,  0.02620751,  0.02438718, -0.14121257, -0.05770359,
         0.01544628,  0.14287737,  0.11075833,  0.20310426,  0.08627865,
        -0.03475839, -0.2666585 , -0.0482914 , -0.11744819,  0.13041186,
         0.01486334,  0.13457212, -0.02613166,  0.24559373, -0.1510532 ,
         0.05351738, -0.08953587, -0.3227936 , -0.25935933,  0.15485485,
         0.15480506,  0.02465698, -0.07033521, -0.5

## Оптимизация

В процессе работы над модулем, а также тестирования моделей, я столкнулся с некоторыми проблемами с оптимизацией моделей, которые я решил следующими изменениями:

**1. Суммаризация - CUDA Out Of Memory**

При инференсе модели суммаризации, я столкнулся с проблемой нехватки памяти (например, при слишком большой длинне текста)

- Поставил порог на колво символов при вводе текста:
```py
class MyGUI():
        def run(self):
...
        # поля для ввода текста
        with self.col1:
            self.text_area1 = st.text_area('**Введите первую статью**', height=150, max_chars=10000)
...
```

- Поставил порог на кол-во символов при суммаризации текста, а также чистку кэша:
```py
class Realization():
    def get_summary(self,
                    text: str, 
                    show_output: bool = False) -> str:
...
        # обрабатываем входящий текст
        input_text = self.get_input(text[10000])
...
        # освобождаем память перед генерацией
        torch.cuda.empty_cache()
...
```

## Разработка программного продукта

Для создания `API`, я буду использовать библиотеку `streamlit`. Это удобная библиотека для быстрого создания и развертывания приложений на `python`

Также, я разделю `API` на `реализационную` и `абстракционную` части, для того чтобы приложение было более структурированным 

### Реализация

Эта часть реализована в файле `backend.py`

```py
# импортирование библиотек
# для работы с cuda
import torch

# для модели суммаризации и токенизатора
from transformers import T5Tokenizer, T5ForConditionalGeneration
# для модели сравнения
from gensim.models.doc2vec import Doc2Vec

# библиотека для nlp
import nltk
# модуль со стоп словами
from nltk.corpus import stopwords
# токенизатор слов
from nltk import word_tokenize
# для лемматизации
from pymystem3 import Mystem

# косинусное сходство
from sklearn.metrics.pairwise import cosine_similarity

# для передачи функции как параметр
from typing import Callable

# для работы с датафреймом
import pandas as pd

# для работы с файлами
import os

# для работы с временем
from time import sleep

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

# создаем класс суммаризации
class Realization:
    # инициализатор
    # -------------------------------------------------------------------------- 
    def __init__(
            self,
            summarization_path: str,
            similiarity_path: str,
            df_path: str,
            device: torch.device | str | None = None,
    ):
        # обработка девайса
        # если девайс задан:
        if device:
            # проверяем на формат
            match type(device):
                case str():
                    # проверка значения переменной
                    if device == 'cpu' or device == 'cuda':
                        # если значение подходит
                        self.device = torch.device(device)
                    # если значение не подходит:
                    else:
                        raise ValueError(f'Устройство (device) должен быть "cuda" или "cpu", не {device}')
                case torch.device:
                    self.device = device
        # если девайс не задан
        else:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # обработка неправильных значений путя к модели суммаризации
        # проверка формата
        if not isinstance(summarization_path, str):
            raise TypeError(f'model_path должен быть в строковом формате, а не {type(model_path)}')
        # проверка путя
        if os.path.exists(summarization_path):
            # пробуем открыть модель и токенизатор
            try:
                # получаем модель
                self.summarization = T5ForConditionalGeneration.from_pretrained(summarization_path).to(device)
                # получаем токенизатор
                self.tokenizer = T5Tokenizer.from_pretrained(summarization_path)
            except Exception as e:
                raise RuntimeError(f'Не удалось загрузить модель или токенизатор: {e}')
        # если путя не существует
        else:
            raise FileNotFoundError(f'Путь {summarization_path} не был найден, попробуйте другой')
        
        # обработка неправильных значений путя к модели сравнения
        # проверка формата
        if not isinstance(similiarity_path, str):
            raise TypeError(f'model_path должен быть в строковом формате, а не {type(model_path)}')
        # проверка путя
        if os.path.exists(similiarity_path):
            # пробуем открыть модель 
            try:
                # получаем модель
                self.similarity = Doc2Vec.load(similiarity_path)

            except Exception as e:
                raise RuntimeError(f'Не удалось загрузить модель: {e}')
        # если путя не существует
        else:
            raise FileNotFoundError(f'Путь {similiarity_path} не был найден, попробуйте другой')

        # обрабатываю путь к датафрейму
        # проверка на формат
        if not isinstance(df_path, str):
            raise TypeError(f'model_path должен быть в строковом формате, а не {type(model_path)}')
        # если неверный формат
        else:
            # проверка на существование
            if os.path.exists(df_path):
                # пробуем получить дф
                try:
                    # получаем датафрейм
                    self.dataframe = pd.read_pickle(df_path)
                except Exception as e:
                    raise RuntimeError(f'Не удалось загрузить датафрейм с эмбеддингами')
            # если нет
            else:
                raise FileNotFoundError(f'Путь {df_path} не был найден, попробуйте другой')
        
        # получаем стоп слова
        self.stop_words = set(stopwords.words('russian'))
        # получаем объект лемматизации
        self.mystem = Mystem()
    # --------------------------------------------------------------------------   
    # метод для чистки стоп слов
    def clean_text(self, text: str) -> str:
        '''
        Функция для чистки текста от стоп слов
            Args:
                - test (str): текст, который нужно почистить от стоп слов
                - stop_words (set): множество стоп слов
            Returns:
                - отфильтрованный текст (str)
        '''
        # создаем список слов используя split()
        words = text.split()
        # фильтруем слова используя заданное множество
        filtered_words = [word for word in words if word.lower() not in self.stop_words]
        # возвращаем почищенный текст
        return ' '.join(filtered_words)

    # --------------------------------------------------------------------------
    # метод для соединения абзацев текста
    @staticmethod
    def remove_empty_line(text: str) -> str:
        '''
        Функция для удаления пустых слов в тексте
            Args: 
                - text (str): текст для убирания пустых строк
            Returns:
                - str: текст с убранными пустыми строками
        '''
        # разделяем текст на строки и оставляем только не пустые строки
        lines = [line for line in text.splitlines() if line.strip() != '']
        # возвращаем новый текст
        return ' '.join(lines)
    
    # функция для создания потокового текста
    @staticmethod
    def stream_data(text: str):
        '''Функция для создания потокового текста'''
        for word in text.split():
            yield word + ' '
            sleep(0.02)

    # функция для лемматизации предложения
    def text_lemmatize(self, text: str) -> str:
        '''
        Функция для лемматизации текста
            Args:
                - text (str): текст в строковом формате

            Returns:
                - processed_text (str): обработанный текст в строковом формате
        
        Пример использования:

        ```python
        >>  from pymystem3 import Mystem
        >> 
        >> # объект Mystem
        >> mystem = Mystem()
        >> 
        >>  print(text_lemmatize('мама мыла раму', mystem))
        ```
        ```Вывод:
        мама мыть рама
        ```
        '''
        # объект лемматизации

        return ''.join(self.mystem.lemmatize(text.strip()))
    
    # --------------------------------------------------------------------------
    # функция для обработки входящего текста
    def get_input(self, text: str) -> str:
        '''Функция для обработки входящего текста'''
        # убираю пустые строки
        text = self.remove_empty_line(text).lower()
        # убираю стоп-слова и лемматизирую
        text = self.text_lemmatize(
            self.clean_text(text)
            )
        # возвращаю обработанный текст
        return text
    
    # --------------------------------------------------------------------------
    # функция для суммаризации
    def get_summary(self,
                    text: str, 
                    show_output: bool = False) -> str:
        '''
        **Функция для суммаризации текста**
        ==
            **Args:**
                - **text** (`str`): входной текст для суммаризации 
                - **show_output** (`bool`): метка для показа результатов внутри функции
            
            **Returns:**
                - **str**: суммаризация входного текста


        Пример использования:
        ====

        ```python
        >>> from function import get_summary
        >>> 
        >>> model = T5ForConditionalGeneration.from_pretrained("./saved_model")
        >>> tokenizer = T5Tokenizer.from_pretrained("./saved_model")
        >>> 
        >>> text = "yout text for summary"
        >>> 
        >>> get_summary(text=text,
        >>>             tokenizer=tokenizer,
        >>>             model=model,
        >>>             device='cuda',
        >>>             show_output=True,
        >>>             )
        >>>             
        >>>  # данный код выведет суммаризацию для вашего текста
            
        '''
        # обработка ошибок
        # если неправильно передан текст
        if not isinstance(text, str):
            raise TypeError(f'Текст (text) должен быть в формате str. Сейчас: {type(text)}')
        
        # если неправильно передана метка показа результатов
        if not isinstance(show_output, bool):
            raise TypeError(f'show_output должен быть в формате bool (True или False). Сейчас: {type(show_output)}')

        # обрабатываем входящий текст
        input_text = self.get_input(text)
        # получаем токены входящего текста
        input_ids = self.tokenizer(input_text, return_tensors='pt').input_ids.to(self.device)
        # генерируем суммаризацию
        outputs = self.summarization.generate(
                                input_ids=input_ids,        # токенизированый входной текст
                                max_length=100,             # максимальная длина генерированной последовательности
                                min_length=10,              # минимальная длина генерированной последовательности
                                num_beams=2,                # количество лучей для поиска с использованием Beam Search
                                do_sample=False,            # ограничивает вариативность
                                repetition_penalty=1.2,     # штраф за повторение токенов
                                no_repeat_ngram_size=3,     # запрещает повторение n-грамм указанного размера
                                num_return_sequences=1,     # количество возвращаемых последовательностей
                                early_stopping=True,        # генерация остановится, как только завершаться гипотезы
                                use_cache=True,             # использование кэширования для ускорения генерации
                                length_penalty=0.8,         # штраф за длину в beam search
                            )
        # декодируем получившийся текст
        gen_summary = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        # выводим результат (если указано)
        if show_output:
            # вывод входного текста
            print(f'>> Original Text: {text}')
            # вывод сгенерированного изложения
            print(f'\n\n>> Generated summary: {gen_summary}')
            # возвращем суммаризацию
            return gen_summary
        
        # если нет, возвращаем суммаризацию без вывода
        else:
            return gen_summary
        

    # функция для сравнения двух текстов
    def get_similarity(
            self,
            text1: str, 
            text2: str, 
            prep_flag: bool = True, 
            preprocess: Callable[[str], str] = get_input
                    ):
        '''
        Функция для получения схожести двух текстов
        ===
            Args:
                - text1 (str): первый текст в строковом формате
                - text2 (str): второй текст в строковом формате
                - prep_flag (bool = True): флаг, нужна ли обработка входного текста
                - preprocess (func(str) -> str: = lambda x: x): функция для обработки текста

            Returns:
                - float: сходство между текстами (от 1 до -1)
        '''
        # проверка формата текста 1
        if not isinstance(text1, str):
            raise TypeError(f'text1 должен быть в строковом формате, а не {type(text1)}')
        # проверка формата текста 2
        if not isinstance(text2, str):
            raise TypeError(f'text1 должен быть в строковом формате, а не {type(text1)}')
        # проверка формата флага для обработки
        if not isinstance(prep_flag, bool):
            raise TypeError(f'prep_flag должен быть в булевом формате, а не {type(prep_flag)}')
        # проверка формата функции обработки текста
        if not isinstance(preprocess, Callable):
            raise TypeError(f'preprocess должна быть ф-ей, принимающей и возвращающей str')

        # если стоит метка о обработке данных
        if prep_flag:
            text1 = preprocess(self, text1)
            text2 = preprocess(self, text2)
        # вычисляем эмбеддинг
        inferred_vector1 = self.similarity.infer_vector(word_tokenize(text1)).reshape(1,-1)
        inferred_vector2 = self.similarity.infer_vector(word_tokenize(text2),).reshape(1,-1)
        # # получаем сходство
        return cosine_similarity(inferred_vector1, inferred_vector2).item()
    

    # функция для поиска топ 3 схожих статей
    def find_top_similar(
            self,
            df: pd.DataFrame, 
            text: str, 
            top_n: int = 3
            ) -> pd.DataFrame:
        '''
        Находит топ-N наиболее схожих эмбеддингов и их summary.

        Параметры:
            - df (pd.DataFrame): Датафрейм с колонками 'summary' и 'embedding'.
            - input_embedding (np.ndarray): Входной эмбеддинг для сравнения.
            - top_n (int): Количество наиболее схожих результатов (по умолчанию 3).

        Возвращает:
            pd.DataFrame: Датафрейм с топ-N наиболее схожими эмбеддингами и их summary.
        '''
        # получаем эмбеддинг
        input_embedding = self.similarity.infer_vector(word_tokenize(self.get_input(text))).reshape(1,-1)
        # список всех схожестей
        similarities = []
        # проходимся по всему датафрейму
        for i in range(df.shape[0]):
            similarities.append(cosine_similarity(input_embedding, df['embedding'].loc[i]).item()*100)
        # Добавляем столбец с косинусной схожестью в датафрейм
        df['similarity'] = similarities
        
        # Сортируем датафрейм по убыванию схожести и выбираем топ-N
        top_similar = df.sort_values(by='similarity', ascending=False).head(top_n)
        summaries = top_similar['summary'].tolist()
        top_similarities = [round(sim, 2) for sim in top_similar['similarity'].tolist()]
        headers = [' '.join(word_tokenize(summary)) for summary in summaries]

        # Возвращаем только нужные колонки
        return top_similarities, headers
```

### Абстракция

Эта часть реализована в файле `app.py`

```py
# ----------------------------------------------
# импортирование библиотек
# для реализации интерфейса
import streamlit as st
# для работы с cuda
import torch
# объект реализации
from backend import Realization
# для логирования времени
from time import time

summarization_path = 'Module2/saved_model'      # путь к модели суммаризации
similarity_path = 'Module2/doc2vec.model'       # путь к модели сравнения
df_path = 'Module2/all_embs.pkl'                # путь к датафрейму с эмбеддингами
# девайс (CUDA или CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class MyGUI:
    # метод инициализации
    def __init__(self):
        '''
        Иициализация приложения
        '''
        # устройство вычисления
        self.device = device
        # объект реализации
        self.realizer = Realization(summarization_path, similarity_path, df_path, device)
        # заголовок приложения
        self.title = st.title(r'$$\text{Приложение для сравнения статей}$$', 
                              help='Решение тестового задания для подготовки к чемпионату "Профессионалы"')
        # разделение на 2 колонки
        self.col1, self.col2 = st.columns(spec=[0.5, 0.5], gap='large', border=True)
        # переменные для хранения текста 
        self.text_area1 = None
        self.text_area2 = None
        # для хранения результатов
        self.view_results1 = None
        self.view_results2 = None
        # для кнопки
        self.compare_button = None
        # для хранения результата сравнения
        self.similarity = None

    def run(self):
        '''
        Основной метод для запуска приложения
        '''
        # поля для ввода текста
        with self.col1:
            self.text_area1 = st.text_area('**Введите первую статью**', height=150)
        # поля для ввода текста
        with self.col2:
            self.text_area2 = st.text_area('**Введите вторую статью**', height=150)

        # кнопка сравнения
        self.compare_button = st.button('**Сравнить**')

        # обработка нажатия кнопки
        if self.compare_button:
            # запоминаем стартовое время
            start_time = time()
            # проверяем заполнены ли обе статьи
            if self.text_area1 and self.text_area2:
                # выводим значок загрузки
                with st.spinner('Обработка...'):
                    # вычисляем суммаризацию
                    summary1 = self.realizer.get_summary(self.text_area1)
                    summary2 = self.realizer.get_summary(self.text_area2)

                # вывод результатов
                with self.col1:
                    # выводим суммаризацию
                    st.write_stream(self.realizer.stream_data(summary1))
                    # выводим топ 3 наиболее похожих статей
                    top_sims, headers = self.realizer.find_top_similar(
                        self.realizer.dataframe,
                        summary1,
                        self.realizer.similarity
                    )
                    # визуализируем результат
                    st.bar_chart(dict(zip(headers, top_sims)))


                with self.col2:
                    # выводим суммаризацию
                    st.write_stream(self.realizer.stream_data(summary2))
                    # выводим топ 3 наиболее похожих статей
                    top_sims, headers = self.realizer.find_top_similar(
                        self.realizer.dataframe,
                        summary2,
                        self.realizer.similarity
                    )
                    # визуализируем результат
                    st.bar_chart(dict(zip(headers, top_sims)))
                    
                # подсчет сходства
                self.similarity = self.realizer.get_similarity(summary1, summary2)
                st.write(f'Схожесть статей: {self.similarity*100:.2f}%')
                if self.similarity:
                    # Столбчатая диаграмма
                    st.bar_chart(
                        [[self.similarity*100, 100]], 
                        x_label='Схожесть', 
                        color=['#4bd4ff', '#0E1117'], 
                        stack="layered", 
                        horizontal=True, 
                        use_container_width=True
                )
                # вывод затраченного времени
                st.write_stream(self.realizer.stream_data(f'Время инференса: {(time()-start_time):.2f}s'))

            # если введены не оба текста
            else: 
                # выводим предупреждения
                st.warning('Пожалуйста, введите текст в оба поля.', icon='⚠️')

# Запуск приложения
if __name__ == '__main__':
    app = MyGUI()
    app.run()
```

Для запуска приложения, надо прописать в терминале следующую команду:

*допустим, сейчас вы находитесь в директории `~\NlP-summary\Module2>`*
```bash
>>> (.venv) ~\NlP-summary\Module2> streamlit run API/app.py
```

**`TODO`**
- ~~сделать анализ эффективности модель~~
- ~~расписать выбор алгоритма~~
- ~~проверить лишние ячейки~~
- ~~расписать оптимизацию~~ 50\50
- ~~расписать создание API~~
- написать документацию
- презентация